<a href="https://colab.research.google.com/github/jazaineam1/BigData2026/blob/main/Cuadernos/7_Elasticsearch_BM25_Compras_Claras.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Abrir S7 en Google Colab"/></a>

# Sesión 7 — Del vecindario al texto: ¿qué procesos debe leer Laura primero?

**Universidad Central · Maestría en Analítica de Datos · Big Data**

**Presentación principal:** https://jazaineam1.github.io/BigData2026/Presentaciones/s07-del-vecindario-al-texto.html  
**Laboratorio guiado:** https://jazaineam1.github.io/BigData2026/assets/tutoriales/s07-laboratorio-guiado.html

En S6 aprendiste a leer un grafo en Neo4j y Cypher. Hoy **no suponemos** que sabes motores de búsqueda, JSON ni Elasticsearch.

> **Pregunta profesional.** Laura ya encontró un vecindario contractual. Ahora tiene muchos procesos para leer. ¿Cómo decide cuáles leer primero si busca, por ejemplo, procesos de **mantenimiento de aeronaves**?

**PARA LLEVAR.** Coincidencia textual no es lo mismo que relevancia ordenada.

## Lo que sí debes poder explicar al terminar

Sin memorizar fórmulas ni volverte administrador de Elastic, deberías poder decir:

1. qué es un **documento**, un **campo** y un **índice**;
2. por qué una búsqueda literal con `contains()` no ordena relevancia;
3. cómo una frase se convierte en términos buscables;
4. qué es la intuición de un **índice invertido**;
5. por qué BM25 combina repetición, rareza y longitud;
6. por qué `descripcion` es `text` pero `id_proceso` es `keyword`;
7. qué hace `match`, qué hace `multi_match` y qué hace un filtro;
8. por qué un score alto no demuestra irregularidad.

## Rúbrica de la práctica S07

Lee esta rúbrica **antes de ejecutar el laboratorio**. La práctica vale 100 puntos. No se evalúa memorizar la fórmula de BM25 ni escribir infraestructura desde cero: se evalúa que puedas **buscar, comparar e interpretar**.

| Criterio | Evidencia completa | Parcial | Sin evidencia | Peso |
|---|---|---|---|---:|
| Punto de partida | declara si usa el JSONL propio de S6 o el respaldo y conserva ID/URL | usa datos pero no declara origen | corpus no rastreable | 10 |
| Comprensión de búsqueda | distingue identificación, filtro y búsqueda; explica por qué `contains()` no ordena | distingue dos acciones | trata todo como equivalente | 15 |
| Ejecución BM25 | top 5 reproducible con consulta y score | ranking incompleto | no produce ranking | 20 |
| Elasticsearch | interpreta `text`/`keyword`, `match`/`multi_match` y filtro; declara motor real | interpretación parcial | afirma ejecutar un motor que no ejecutó | 15 |
| Experimento A/B | cambia una sola decisión y explica el efecto | mezcla cambios | no compara | 15 |
| Decisión profesional | elige qué leer primero con evidencia propia | elige sin justificar | copia el primer resultado | 10 |
| Dudoso y límite | identifica falso positivo/dudoso y qué no demuestra el ranking | falta uno de los dos | interpreta score como irregularidad | 10 |
| Trazabilidad | entrega CSV + hito con autor/alias y motor | falta un producto | no deja evidencia | 5 |

## Mapa de la sesión

| Bloque | Pregunta | Acción |
|---|---|---|
| Recuperar | ¿qué procesos dejó S6? | cargar JSONL o respaldo |
| Distinguir | ¿identificar, filtrar y buscar son lo mismo? | tres mini consultas |
| Comparar | ¿qué limita `contains()`? | línea base |
| Indexar | ¿cómo pasa el texto a términos? | tokens e índice invertido |
| Ordenar | ¿por qué aparece uno primero? | BM25 local |
| Traducir | ¿cómo se ve en Elasticsearch? | mapping y consultas simples |
| Decidir | ¿qué leería Laura y qué no puede afirmar? | CSV + hito |

---
# 1. Recuperar los procesos

S6 debía producir `s06_contexto_procesos.jsonl`. Si está en Colab, lo usamos. Si no está, usamos un **respaldo versionado** con procesos reales del curso.

**OJO.** El respaldo permite aprender búsqueda, pero no sustituye la evidencia propia de S6.

In [ ]:
from pathlib import Path
import json, re, math, unicodedata
from collections import Counter, defaultdict
import pandas as pd
import numpy as np
from IPython.display import display

ARCHIVO_S6 = Path("s06_contexto_procesos.jsonl")
RESPALDO = json.loads(r'''[{"id_proceso":"CO1.REQ.3755377","entidad":"FUERZA AEROESPACIAL COLOMBIANA","nombre_proceso":"Mantenimiento programado y no programado de aeronaves KFIR","descripcion":"SERVICIOS DE MANTENIMIENTO PROGRAMADO (RUTINA) Y MANTENIMIENTO NO PROGRAMADO (NO RUTINA) DE LAS AERONAVES KFIR DE LA FUERZA AÉREA COLOMBIANA; DE ACUERDO A ANEXO TÉCNICO","modalidad":"Contratación Directa (con ofertas)","url_secop":"https://community.secop.gov.co/Public/Tendering/OpportunityDetail/Index?noticeUID=CO1.NTC.3675471","motor_ejecucion":"corpus versionado de respaldo"},{"id_proceso":"CO1.REQ.3714823","entidad":"FUERZA AEROESPACIAL COLOMBIANA","nombre_proceso":"Inspección estructural PDM de aeronave C-130 FAC1018","descripcion":"EL SERVICIO DE MANTENIMIENTO INSPECCIÓN MAYOR ESTRUCTURAL PDM; CUMPLIMIENTO DE BOLETINES DE SERVICIO MANDATORIOS Y TRABAJOS ESPECIALES DE LA AERONAVE C-130 FAC1018; DE ACUERDO CON ANEXO TÉCNICO","modalidad":"Contratación Directa (con ofertas)","url_secop":"https://community.secop.gov.co/Public/Tendering/OpportunityDetail/Index?noticeUID=CO1.NTC.3636264","motor_ejecucion":"corpus versionado de respaldo"},{"id_proceso":"CO1.REQ.3730074","entidad":"FUERZA AEROESPACIAL COLOMBIANA","nombre_proceso":"Construcción fase II centro de instrucción e investigación aeronáutica","descripcion":"CONSTRUCCION DE LA FASE II DEL CENTRO DE INSTRUCCIÓN EN CAÍDA LIBRE E INVESTIGACIÓN AERONÁUTICA DE LA FAC EN LAS INSTALACIONES DEL MUSEO AEROESPACIAL EN TOCANCIPÁ - CUNDINAMARCA","modalidad":"Contratación Directa (con ofertas)","url_secop":"https://community.secop.gov.co/Public/Tendering/OpportunityDetail/Index?noticeUID=CO1.NTC.3644540","motor_ejecucion":"corpus versionado de respaldo"},{"id_proceso":"CO1.REQ.8018431","entidad":"FUERZA AEROESPACIAL COLOMBIANA","nombre_proceso":"Adquisición de blindajes aeronáuticos","descripcion":"LA ADQUISICIÓN BLINDAJES AERONÁUTICOS; SEGÚN ANEXO TÉCNICO; COMO SOPORTE PARA LA FUNCIONALIDAD Y OPERATIVIDAD DE LOS SISTEMAS DE ARMAS DE LA FUERZA AEROESPACIAL COLOMBIANA.","modalidad":"Contratación directa","url_secop":"https://community.secop.gov.co/Public/Tendering/OpportunityDetail/Index?noticeUID=CO1.NTC.7912562","motor_ejecucion":"corpus versionado de respaldo"},{"id_proceso":"CO1.REQ.9021155","entidad":"MUNICIPIO DE RIONEGRO","nombre_proceso":"Construcción e interventoría de ciudadela educativa","descripcion":"CONTRATO INTERADMINISTRATIVO POR ADMINISTRACIÓN DELEGADA DE RECURSOS PARA LA CONSTRUCCIÓN E INTERVENTORIA DEL PROYECTO CIUDADELA EDUCATIVA 4.0 ETAPA 1 EN EL MUNICIPIO DE RIONEGRO.","modalidad":"Contratación directa","url_secop":"https://community.secop.gov.co/Public/Tendering/OpportunityDetail/Index?noticeUID=CO1.NTC.8886200","motor_ejecucion":"corpus versionado de respaldo"},{"id_proceso":"CO1.REQ.9229871","entidad":"MINISTERIO DEL DEPORTE","nombre_proceso":"Construcción centro de alto rendimiento deportivo","descripcion":"AUNAR ESFUERZOS TECNICOS; ADMINISTRATIVOS Y FINANCIEROS PARA LA EJECUCION DEL PROYECTO DENOMINADO CONSTRUCCIÓN DE UN CENTRO DE ALTO RENDIMIENTO DEPORTIVO EN LA CIUDAD DE MONTERÍA","modalidad":"Contratación directa","url_secop":"https://community.secop.gov.co/Public/Tendering/OpportunityDetail/Index?noticeUID=CO1.NTC.9092192","motor_ejecucion":"corpus versionado de respaldo"},{"id_proceso":"CO1.REQ.1253135","entidad":"FONDO UNICO DE TECNOLOGÍAS DE LA INFORMACIÓN Y LAS COMUNICACIONES","nombre_proceso":"Servicios TIC de emergencia sanitaria","descripcion":"PONER A DISPOSICIÓN DE SUSCRIPTORES LÍNEAS MÓVILES PREPAGO SERVICIOS TIC - EMERGENCIA SANITARIA; ECONÓMICA Y ECOLÓGICA COVID-19 - COMUNICACIÓN CELULAR S.A. COMCEL S.A.","modalidad":"Contratación directa","url_secop":"https://community.secop.gov.co/Public/Tendering/OpportunityDetail/Index?noticeUID=CO1.NTC.1213079","motor_ejecucion":"corpus versionado de respaldo"},{"id_proceso":"CO1.REQ.10017413","entidad":"DISTRITO ESPECIAL INDUSTRIAL Y PORTUARIO DE BARRANQUILLA","nombre_proceso":"Operación de equipos básicos de salud","descripcion":"CONTRATO INTERADMINITRATIVO PARA EL FORTALECIMIENTO DEL NIVEL PRIMARIO; BASADO EN LA ATENCIÓN PRIMARIA EN SALUD A TRAVÉS DE LA CONFORMACIÓN Y OPERACIÓN DE EQUIPOS BÁSICOS DE SALUD","modalidad":"Contratación directa","url_secop":"https://community.secop.gov.co/Public/Tendering/OpportunityDetail/Index?noticeUID=CO1.NTC.9868453","motor_ejecucion":"corpus versionado de respaldo"}]''')

def cargar_jsonl(path):
    return pd.read_json(path, lines=True, dtype=False)

if ARCHIVO_S6.exists():
    corpus = cargar_jsonl(ARCHIVO_S6)
    origen_corpus = "archivo propio de S6"
else:
    corpus = pd.DataFrame(RESPALDO)
    origen_corpus = "respaldo versionado S07"

for col in ["id_proceso","entidad","nombre_proceso","descripcion","modalidad","url_secop","motor_ejecucion"]:
    if col not in corpus:
        corpus[col] = ""
    corpus[col] = corpus[col].fillna("").astype(str)

corpus = corpus.drop_duplicates("id_proceso").reset_index(drop=True)
corpus["texto_busqueda"] = (corpus["nombre_proceso"].str.strip()+" "+corpus["descripcion"].str.strip()).str.strip()

print("Origen:", origen_corpus)
print("Procesos disponibles:", len(corpus))
print("Entidades:", corpus["entidad"].nunique())
display(corpus[["id_proceso","entidad","nombre_proceso","descripcion"]].head(8))

### Subir tu archivo S6 si lo tienes

Ejecuta la siguiente celda solo si tienes `s06_contexto_procesos.jsonl`.

**Qué debe verse:** `Procesos disponibles: N`.  
**Error frecuente:** subir el Markdown del hito en lugar del JSONL.

In [ ]:
#@title Subir s06_contexto_procesos.jsonl (opcional)
USAR_ARCHIVO_PROPIO = False #@param {type:"boolean"}

if USAR_ARCHIVO_PROPIO:
    try:
        from google.colab import files
        subidos = files.upload()
        if "s06_contexto_procesos.jsonl" not in subidos:
            raise ValueError("Sube exactamente s06_contexto_procesos.jsonl")
        corpus = cargar_jsonl("s06_contexto_procesos.jsonl")
        origen_corpus = "archivo propio de S6"
        for col in ["id_proceso","entidad","nombre_proceso","descripcion","modalidad","url_secop","motor_ejecucion"]:
            if col not in corpus:
                corpus[col] = ""
            corpus[col] = corpus[col].fillna("").astype(str)
        corpus = corpus.drop_duplicates("id_proceso").reset_index(drop=True)
        corpus["texto_busqueda"] = (corpus["nombre_proceso"]+" "+corpus["descripcion"]).str.strip()
        print("Procesos disponibles:", len(corpus))
    except ImportError:
        print("Fuera de Colab: copia el JSONL junto al notebook y vuelve a ejecutar.")
else:
    print("Se conserva:", origen_corpus)

---
# 2. Antes de Elasticsearch: tres acciones distintas

| Acción | Pregunta | Ejemplo |
|---|---|---|
| Identificar | ¿cuál es este proceso exacto? | `CO1.REQ.3755377` |
| Filtrar | ¿qué procesos son de esta entidad? | Fuerza Aeroespacial |
| Buscar texto | ¿cuáles hablan de mantenimiento de aeronaves? | consulta textual |

Elasticsearch se vuelve útil cuando el problema no es solo traer filas, sino **ordenar documentos por relevancia textual**.

In [ ]:
# Identificar exactamente por ID
id_ejemplo = corpus.loc[0, "id_proceso"]
display(corpus.loc[corpus["id_proceso"].eq(id_ejemplo), ["id_proceso","nombre_proceso"]])

In [ ]:
# Filtrar por entidad
entidad_ejemplo = "FUERZA AEROESPACIAL COLOMBIANA"
filtro_entidad = corpus[corpus["entidad"].str.upper().eq(entidad_ejemplo)]
print("Procesos de la entidad:", len(filtro_entidad))
display(filtro_entidad[["id_proceso","nombre_proceso"]])

---
# 3. Coincidencia literal: útil, pero insuficiente

`contains()` responde si una cadena aparece. No explica cuál proceso debe leerse primero.

In [ ]:
termino = "mantenimiento"
literal = corpus[corpus["texto_busqueda"].str.contains(termino, case=False, regex=False, na=False)][["id_proceso","nombre_proceso","descripcion"]]
print("Coincidencias literales:", len(literal))
display(literal)

**Cómo se lee.** Si aparece una fila, contiene la cadena `mantenimiento`.

**Qué NO dice.** No distingue si también habla de aeronaves, si la palabra es común o si aparece perdida en un texto largo.

---
# 4. Documento, campo e índice

| Palabra | En esta clase significa |
|---|---|
| Documento | un proceso contractual |
| Campo | una parte del documento: entidad, descripción, URL |
| Índice | una colección preparada para buscar rápidamente |

Piensa en el índice como una biblioteca organizada para encontrar documentos por palabras.

---
# 5. Analyzer: convertir texto en términos buscables

`"Servicios de MANTENIMIENTO de las Aeronaves KFIR"`

puede convertirse en:

`servicios · mantenimiento · aeronaves · kfir`

No necesitas memorizar teoría lingüística. Conserva esta idea: **lo que escribo y lo que el buscador guarda para buscar no son necesariamente la misma cadena**.

In [ ]:
STOP = {"de","la","el","los","las","del","y","en","para","por","un","una","con","a","al","se","su","sus","como","que","es","no","e"}

def tokens_es(texto):
    texto = unicodedata.normalize("NFKD", str(texto))
    texto = "".join(c for c in texto if not unicodedata.combining(c)).lower()
    palabras = re.findall(r"[a-z0-9]+", texto)
    return [p for p in palabras if len(p) > 2 and p not in STOP]

def mostrar_tokens(texto):
    return pd.DataFrame({"terminos_buscables": tokens_es(texto)})

In [ ]:
texto_demo = "Servicios de MANTENIMIENTO de las Aeronaves KFIR"
display(mostrar_tokens(texto_demo))

### Práctica 1 — cambia una frase

Modifica solo el texto entre comillas. Observa qué términos quedan.

In [ ]:
mi_frase = "mantenimiento de equipo aeronáutico"
display(mostrar_tokens(mi_frase))

---
# 6. Índice invertido

En vez de abrir documento por documento cada vez, el buscador conserva algo parecido a:

`mantenimiento → D1, D2`  
`aeronaves → D1`  
`blindajes → D4`

Eso se llama **índice invertido**: va de términos a documentos.

In [ ]:
def construir_indice_invertido(df, campo="texto_busqueda"):
    indice = defaultdict(list)
    for fila, texto in enumerate(df[campo]):
        for termino in sorted(set(tokens_es(texto))):
            indice[termino].append(fila)
    return dict(indice)

indice = construir_indice_invertido(corpus)
consulta_terminos = ["mantenimiento","aeronaves","aeronautica","construccion","blindajes"]
tabla_indice = pd.DataFrame({"termino":consulta_terminos,"documentos_donde_aparece":[indice.get(t,[]) for t in consulta_terminos]})
display(tabla_indice)

**PARA LLEVAR.** El índice invertido localiza candidatos. Todavía falta ordenarlos.

---
# 7. BM25: ordenar candidatos

BM25 combina tres intuiciones:

1. **Repetición con saturación:** aparecer más veces ayuda, pero no crece sin límite.
2. **Rareza:** una palabra presente en pocos documentos discrimina más.
3. **Longitud:** una coincidencia en un texto corto no pesa igual que una coincidencia perdida en uno enorme.

La fórmula existe, pero hoy evaluamos la **interpretación**.

In [ ]:
def preparar_bm25(df, campo="texto_busqueda"):
    docs = [tokens_es(x) for x in df[campo]]
    N = len(docs)
    longitudes = np.array([len(d) for d in docs], dtype=float)
    avgdl = float(longitudes.mean()) if N else 0.0
    df_term = Counter()
    for d in docs:
        df_term.update(set(d))
    return docs, N, longitudes, avgdl, df_term

def buscar_bm25(df, consulta, campo="texto_busqueda", k1=1.2, b=0.75):
    docs, N, longitudes, avgdl, df_term = preparar_bm25(df, campo)
    query_tokens = tokens_es(consulta)
    def idf(t):
        n = df_term.get(t, 0)
        return math.log(1 + (N - n + 0.5)/(n + 0.5)) if N else 0.0
    def score_doc(tokens_doc, dl):
        tf = Counter(tokens_doc); score = 0.0
        for t in query_tokens:
            f = tf.get(t, 0)
            if f == 0: continue
            denom = f + k1*(1-b+b*(dl/avgdl if avgdl else 0))
            score += idf(t)*(f*(k1+1))/denom
        return score
    scores = [score_doc(doc, longitudes[i]) for i,doc in enumerate(docs)]
    out = df.copy(); out["score_bm25"] = scores
    out = out[out["score_bm25"]>0].sort_values(["score_bm25","id_proceso"],ascending=[False,True]).reset_index(drop=True)
    out.insert(0,"rank",range(1,len(out)+1))
    return out

def explicar_bm25(df, consulta, id_proceso):
    docs,N,longitudes,avgdl,df_term = preparar_bm25(df)
    fila = int(df.index[df["id_proceso"].eq(id_proceso)][0])
    tf = Counter(docs[fila]); filas=[]
    for t in tokens_es(consulta):
        n=df_term.get(t,0); idf=math.log(1+(N-n+0.5)/(n+0.5)) if N else 0.0; f=tf.get(t,0)
        den=f+1.2*(1-0.75+0.75*(longitudes[fila]/avgdl if avgdl else 0))
        aporte=idf*(f*2.2)/den if f else 0
        filas.append({"termino":t,"apariciones":f,"documentos_que_lo_contienen":n,"rareza_idf":round(idf,4),"aporte_al_score":round(aporte,4)})
    return pd.DataFrame(filas)

### Práctica 2 — busca y predice

Antes de ejecutar, predice qué procesos deberían subir para `mantenimiento aeronaves`.

In [ ]:
consulta_A = "mantenimiento aeronaves"
ranking_A = buscar_bm25(corpus, consulta_A)
display(ranking_A[["rank","id_proceso","nombre_proceso","score_bm25","url_secop"]].head(5))

### Explica el primer resultado

Un ranking sin lectura se vuelve una caja negra.

In [ ]:
if len(ranking_A)==0:
    raise ValueError("La consulta no produjo resultados. Cambia consulta_A.")
id_top = ranking_A.loc[0,"id_proceso"]
print("Documento explicado:", id_top)
display(explicar_bm25(corpus, consulta_A, id_top))

---
# 8. De BM25 local a Elasticsearch

Elasticsearch añade una plataforma real para crear un **índice**, definir tipos con un **mapping**, analizar texto, ejecutar consultas y devolver `_score` y fragmentos resaltados.

No necesitas administrar un cluster hoy. Necesitas entender qué pregunta le haces.

## `text` o `keyword`

| Campo | ¿Buscar palabras dentro? | Tipo |
|---|---:|---|
| `descripcion` | sí | `text` |
| `nombre_proceso` | sí | `text` |
| `id_proceso` | no | `keyword` |
| `entidad` | no; filtro exacto | `keyword` |
| `modalidad` | no; filtro/agrupación | `keyword` |

**Regla simple.** Si buscas palabras dentro, piensa en `text`. Si quieres igualdad exacta, piensa en `keyword`.

In [ ]:
mapping_s07 = {
    "mappings":{"properties":{
        "id_proceso":{"type":"keyword"},
        "entidad":{"type":"keyword"},
        "modalidad":{"type":"keyword"},
        "nombre_proceso":{"type":"text","analyzer":"spanish"},
        "descripcion":{"type":"text","analyzer":"spanish"},
        "url_secop":{"type":"keyword","index":False}
    }}
}
mapping_s07

## Cómo leer una consulta JSON

```json
{
  "match": {
    "descripcion": "mantenimiento aeronaves"
  }
}
```

Se lee: **usa match → en el campo descripcion → con el texto mantenimiento aeronaves**.

In [ ]:
consulta_elastic_sencilla = {"query":{"match":{"descripcion":"mantenimiento aeronaves"}}}
consulta_elastic_sencilla

## Escalera de consultas

1. `match`: buscar texto en un campo.
2. `multi_match`: buscar en varios campos.
3. `filter`: restringir sin cambiar el score.
4. `highlight`: mostrar dónde coincidió.

Elasticsearch devuelve `_score`. Ese score es relevancia textual, no riesgo ni verdad jurídica.

In [ ]:
consulta_elastic_profesional = {
    "query":{"bool":{
        "must":[{"multi_match":{"query":"mantenimiento aeronaves","fields":["nombre_proceso^2","descripcion"]}}],
        "filter":[{"term":{"entidad":"FUERZA AEROESPACIAL COLOMBIANA"}}]
    }},
    "highlight":{"fields":{"nombre_proceso":{},"descripcion":{}}},
    "size":5
}
consulta_elastic_profesional

---
# 9. Arquitectura: solo lo necesario

- un **cluster** reúne recursos de Elasticsearch;
- un **índice** se divide en **shards**;
- cada documento pertenece a un **primary shard**;
- una **replica shard** aporta redundancia y capacidad de lectura.

En Elastic Cloud Serverless, esta infraestructura se automatiza. Hoy la estudiamos como modelo mental, no como configuración.

---
# 10. Ruta profesional opcional: Elastic Cloud

La ruta obligatoria ya funciona con BM25 local. Si el docente tiene un proyecto listo o tú ya tienes Elastic Cloud, ejecuta esta parte.

**No gastes la sesión creando cuentas si falla el acceso.** Declara el motor realmente ejecutado.

In [ ]:
#@title Conectar Elastic Cloud (opcional)
USAR_ELASTIC = False #@param {type:"boolean"}
client = None
motor_elastic = "no ejecutado"

if USAR_ELASTIC:
    import importlib.util, subprocess, sys
    if importlib.util.find_spec("elasticsearch") is None:
        subprocess.check_call([sys.executable,"-m","pip","install","-q","elasticsearch>=9,<10"])
    from getpass import getpass
    from elasticsearch import Elasticsearch
    endpoint = input("Elasticsearch endpoint (https://...): ").strip()
    api_key = getpass("Elasticsearch API key: ").strip()
    client = Elasticsearch(endpoint, api_key=api_key, request_timeout=30)
    info = client.info()
    motor_elastic = "Elasticsearch "+info.get("version",{}).get("number","?")
    print("Conexión verificada:", motor_elastic)
else:
    print("Elastic Cloud omitido. Continúa con BM25 local.")

In [ ]:
#@title Crear índice y cargar documentos (solo si conectaste)
INDEX_NAME = "s07_compras_claras"
if client is not None:
    from elasticsearch.helpers import bulk
    if client.indices.exists(index=INDEX_NAME):
        client.indices.delete(index=INDEX_NAME)
    client.indices.create(index=INDEX_NAME, **mapping_s07)
    acciones=[]
    for d in corpus.to_dict("records"):
        fuente={k:d.get(k,"") for k in ["id_proceso","entidad","modalidad","nombre_proceso","descripcion","url_secop"]}
        acciones.append({"_index":INDEX_NAME,"_id":d["id_proceso"],"_source":fuente})
    ok,errores = bulk(client,acciones,refresh=True,raise_on_error=False)
    print("Documentos indexados:",ok,"| errores:",len(errores))
else:
    print("No hay conexión Elastic. La ruta local sigue completa.")

In [ ]:
#@title Ejecutar multi_match en Elastic
if client is not None:
    resp=client.search(index=INDEX_NAME, **consulta_elastic_profesional)
    filas=[]
    for h in resp["hits"]["hits"]:
        s=h["_source"]
        filas.append({"id_proceso":s.get("id_proceso"),"score":h.get("_score"),"entidad":s.get("entidad"),"nombre_proceso":s.get("nombre_proceso"),"highlight":" ... ".join(sum(h.get("highlight",{}).values(),[])),"url_secop":s.get("url_secop")})
    elastic_resultados=pd.DataFrame(filas); display(elastic_resultados)
else:
    elastic_resultados=pd.DataFrame(); print("Consulta Elastic preparada, pero no ejecutada.")

---
# 11. Práctica 3 — experimento controlado

Compara dos consultas. Cambia **una sola cosa** para poder explicar el efecto.

In [ ]:
consulta_B = "mantenimiento aeronautico"
ranking_B = buscar_bm25(corpus, consulta_B)

print("Consulta A:", consulta_A)
display(ranking_A[["rank","id_proceso","nombre_proceso","score_bm25"]].head(5))
print("Consulta B:", consulta_B)
display(ranking_B[["rank","id_proceso","nombre_proceso","score_bm25"]].head(5))

---
# 12. Hito S07

Tu evidencia debe responder:

- qué necesitaba buscar Laura;
- qué consulta usaste;
- qué proceso leerías primero;
- qué cambió en el experimento;
- qué resultado fue dudoso;
- qué alternativa descartaste;
- qué límite tiene este ranking.

In [ ]:
from datetime import datetime, timezone
from pathlib import Path

AUTOR_ALIAS = input("Autor o alias del equipo: ").strip()
NECESIDAD = input("Necesidad de búsqueda de Laura: ").strip()
DECISION = input("¿Cuál proceso leerías primero y por qué? ").strip()
CAMBIO = input("¿Qué cambiaste entre A y B y qué efecto observaste? ").strip()
DUDOSO = input("Identifica un resultado dudoso/falso positivo y explica por qué: ").strip()
ALTERNATIVA = input("Alternativa descartada y razón: ").strip()
LIMITE = input("Límite concreto: qué NO demuestra este ranking y qué dato faltaría: ").strip()

campos=[AUTOR_ALIAS,NECESIDAD,DECISION,CAMBIO,DUDOSO,ALTERNATIVA,LIMITE]
if min(map(len,campos))<10:
    raise ValueError("Completa cada respuesta con una frase específica.")
fecha_utc=datetime.now(timezone.utc).isoformat()
print("Registro interpretativo listo:",fecha_utc)

In [ ]:
top5 = ranking_A.head(5).copy()
top5["consulta"]=consulta_A
top5["motor"]="BM25 local"
cols=["rank","id_proceso","entidad","nombre_proceso","score_bm25","url_secop","consulta","motor"]
top5[cols].to_csv("s07_resultados_busqueda.csv",index=False,encoding="utf-8-sig")

motor_real="BM25 local"
if client is not None:
    motor_real += " + " + motor_elastic

hito=f"""# Hito S07 — Relevancia textual

- Autor/alias: {AUTOR_ALIAS}
- Fecha UTC: {fecha_utc}
- Origen del corpus: {origen_corpus}
- Procesos disponibles: {len(corpus)}
- Necesidad de búsqueda: {NECESIDAD}
- Consulta A: {consulta_A}
- Consulta B: {consulta_B}
- Motor realmente ejecutado: {motor_real}

## Top 5 — consulta A

{top5[["rank","id_proceso","nombre_proceso","score_bm25","url_secop"]].to_markdown(index=False)}

## Decisión
{DECISION}

## Cambio ensayado
{CAMBIO}

## Resultado dudoso / falso positivo
{DUDOSO}

## Alternativa descartada
{ALTERNATIVA}

## Límite
{LIMITE}

### Recordatorio
La relevancia textual ordena documentos respecto de una consulta; por sí sola no demuestra irregularidad, riesgo, causalidad ni importancia jurídica.
"""
Path("hito_s07_relevancia.md").write_text(hito,encoding="utf-8")
print(hito)
print("\nArchivos creados: s07_resultados_busqueda.csv + hito_s07_relevancia.md")

In [ ]:
#@title Descargar evidencia en Colab
try:
    from google.colab import files
    files.download("s07_resultados_busqueda.csv")
    files.download("hito_s07_relevancia.md")
except ImportError:
    print("Fuera de Colab: los archivos quedaron en el directorio de trabajo.")

## Comprobación final

- el CSV tiene ranking, score, URL, consulta y motor;
- el hito declara si usaste archivo propio de S6 o respaldo;
- hay una decisión y una alternativa descartada;
- nombras un resultado dudoso;
- escribes un límite claro: **relevancia textual no demuestra irregularidad**.

## Cierre

| Pregunta | Herramienta |
|---|---|
| ¿Cómo represento documentos? | MongoDB |
| ¿Cómo sirvo una consulta conocida? | Cassandra |
| ¿Qué relaciones rodean el proceso? | Neo4j |
| ¿Qué texto parece más relevante? | Elasticsearch / BM25 |

S8 será el primer taller de evaluación. S9 permitirá contrastar relevancia **léxica** con relevancia **semántica**.

## Referencias oficiales verificadas · 23-sep-2026

- Elastic Similarity / BM25: https://www.elastic.co/docs/reference/elasticsearch/mapping-reference/similarity
- Text analysis: https://www.elastic.co/docs/manage-data/data-store/text-analysis
- Spanish analyzer: https://www.elastic.co/docs/reference/text-analysis/analysis-lang-analyzer
- Keyword fields: https://www.elastic.co/docs/reference/elasticsearch/mapping-reference/keyword
- Term query: https://www.elastic.co/docs/reference/query-languages/query-dsl/query-dsl-term-query
- Multi-match: https://www.elastic.co/docs/reference/query-languages/query-dsl/query-dsl-multi-match-query
- Clusters, nodes and shards: https://www.elastic.co/docs/deploy-manage/distributed-architecture/clusters-nodes-shards
- Highlighting: https://www.elastic.co/docs/reference/elasticsearch/rest-apis/highlighting